# Notebook 1 - Extracting XY Coordinates from Vector Data
`GemGIS` is a Python-based, open-source geographic information processing library.
It is capable of preprocessing spatial data such as vector data (shape files, geojson files,
geopackages), raster data (tif, png,...), data obtained from web services (WMS, WFS, WCS) or XML/KML
files. Preprocessed data can be stored in a dedicated Data Class to be passed to the geomodeling package
`GemPy` (https://github.com/cgre-aachen/gempy) in order to accelerate to model building process. In addition, enhanced 3D visualization of data is
powered by the `PyVista` (https://github.com/pyvista/pyvista) package.

# Content

[Overview](#overview) <br>
[Load Libraries](#libraries) <br>
[What are GeoDataFrames?](#gdfs) <br>
[Example 1 - Point Data](#ex1) <br>
[Example 2 - LineString Data](#ex2) <br>
[Example 3 - Polygon Data](#ex3) <br>
[Example 4 - Extract Data from any type of Geometry](#ex4) <br>
[Summary](#summary) <br>


<a id='overview'></a>
# Overview

This notebook will present how to extract X and Y coordinates from vector data (shape files, geojsons, geopackages) loaded as GeoDataFrame using `GeoPandas` (https://github.com/geopandas/geopandas). This vector data consists of either `Point` data, `LineString` data, `MultiLineString` data or `Polygon` data. We will use a sample geological map to introduce the functionality of the different functions. The data was prepared in QGIS as shown below. 

<img src="../data/tutorial01/qgis.png" width="600">

The aim of this and the upcoming tutorials is to demonstrate how to prepare spatial data for geomodeling with `GemPy` to get a geological model like shown below. 

<img src="../data/tutorial01/cover.png" width="600">

<a id='libraries'></a>
# Load Libraries

Firstly, we import the necessary libraries that we need for the extraction of X and Y coordinates. This includes the `GemGIS` library, `GeoPandas` to work with vector data and `Matplotlib` (https://github.com/matplotlib/matplotlib) for plotting. `Pandas` (https://github.com/pandas-dev/pandas) is needed for the manipulation of tables. 

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import sys
sys.path.append('../../gemgis')
import gemgis as gg
print(gg.__version__)

<a id='gdfs'></a>
# What are GeoDataFrames? 

GeoDataFrames are Pandas DataFrames with an additional `geometry` column. This geometry column is also called a `GeoSeries`. It is a vector where each entry in the vector is a set of shapes corresponding to one observation. An entry may consist of only one shape (like a single polygon) or multiple shapes that are meant to be thought of as one observation (like the many polygons that make up the State of Hawaii or a country like Indonesia).

`GeoPandas` has three basic classes of geometric objects (which are actually `shapely` objects):

- Points / Multi-Points
- Lines / Multi-Lines
- Polygons / Multi-Polygons

Source: https://geopandas.org/data_structures.html

A `GeoDataFrame` has many different attributes. The most important ones are shown below.

As mentioned above, every entry of the `GeoDataFrame` is basically a shapely object, in this case a point:

Depending on the geometry types in your GeoDataFrame other attributes may be available such as `gdf.length` to return the lengths of LineStrings or `gdf.area` to return the area of Polygons.

<a id='ex1'></a>
# Example 1 - Point Data

## Load Data

As a first example to show how to extract X and Y coordinates from vector data, we are loading a shape file containing point information. Additional columns are `id`, which was not recorded when digitizing the data and `formation` indicating the base of the formation that was encountered at a particular point. This will become important in future tutorials. The term `Ton` is German for clay and indicates in this example the boundary between a sandstone layer and a claystone layer. 

## Inspecting the Data

Firstly, we can inspect the data that we have loaded.

As mentioned above, every entry of the `GeoDataFrame` is basically a shapely object, in this case a point:

## Plotting Data

The data can be plotted using the built-in `GeoPandas` plotting functions and by adding additional `Matplotlib` commands like grids or axes labels.

## Extracting XY Coordinates

For the geomodeling with `GemPy` or other packages, the X and Y coordinates of the `shapely` points need to be extracted. The extraction of the X and Y values for points is relatively straight forward. The X and Y coordinates for points can be accessed using `gdf.geometry.x` and `gdf.geometry.y`. This functionality is also combined in the function `extract_xy_points`. 

The signature of the function is shown below:

```Signature:
gg.vector.extract_xy_points(
    gdf: geopandas.geodataframe.GeoDataFrame,
    reset_index: bool = True,
    drop_id: bool = True,
    drop_index: bool = True,
    overwrite_xy: bool = False,
    target_crs: str = None,
    bbox: Union[Sequence[float], NoneType] = None,
) -> geopandas.geodataframe.GeoDataFrame
Docstring:
Extracting x,y coordinates from a GeoDataFrame (Points) and returning a GeoDataFrame with x,y
coordinates as additional columns
Args:
    gdf (gpd.geodataframe.GeoDataFrame): GeoDataFrame created from vector data containing elements of geom_type Point
    reset_index (bool): Variable to reset the index of the resulting GeoDataFrame, default True
    drop_id (bool): Variable to drop the id column, default True
    drop_index (bool): Variable to drop the index column, default True
    overwrite_xy (bool): Variable to overwrite existing X and Y values, default False
    target_crs (str, pyproj.crs.crs.CRS): Name of the CRS provided to reproject coordinates of the GeoDataFrame
    bbox (list): Values (minx, maxx, miny, maxy) to limit the extent of the data
Return:
    gdf (gpd.geodataframe.GeoDataFrame): GeoDataFrame with appended x,y columns and optional columns```

## Extracting X and Y coordinates using GemGIS

In its simplest version, `extract_xy_points` will return a `GeoDataFrame` with Points as geometry type, an additional `X` and `Y`column and the remaining columns of the original `gdf`. The `id` column will be dropped by default as it is usally not used when creating the data in your GIS software.

An argument `reset_index=False` can be passed to prevent resetting the index. However, this will have no effect on `extract_xy_points` unless the data is cropped as shown below.

An argument `drop_id=False` can be passed to prevent dropping the `id` column if it is needed for further processing.

A `target_crs` can be provided with an EPSG code as string or a pyproj CRS object (https://pyproj4.github.io/pyproj/dev/api/crs/crs.html) to reproject the current coordinates into a different Coordinate Reference System (=CRS). As we deal with a Long/Lat coordinate system here, the reprojection will fail. Therefore, we pass the original CRS again.

A bounding box (`bbox`) containing `minx`, `maxx`, `miny`, `maxy` values can be provided to limit the extent of the data. The original `gdf` consists of 41 elements, the cropped `gdf_coordinates` of only 11 elements. Please notice, that the index was automatically reset. Passing `reset_index=False` will prevent resetting the index.

A clipping of the data can also be performed the same way for LineStrings, MultiLineStrings and Polygons but the demonstration of the feature will be limited to Point data here.

The original data and the cropped/clipped data can now be visualized. The red rectangle marks the area that is being kept after the clipping. 

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2)
gdf.plot(ax=ax1, aspect='equal')
ax1.set_xlim(0,1000)
ax1.set_ylim(0,1100)
ax1.grid()

ax1.axhline(700, c='red')
ax1.axhline(150, c='red')
ax1.axvline(600, c='red')
ax1.axvline(850, c='red')
ax1.set_xlabel('X [m]')
ax1.set_ylabel('Y [m]')

gdf_coordinates.plot(ax=ax2, aspect='equal')
ax2.set_xlim(0,1000)
ax2.set_ylim(0,1100)
ax2.grid()

ax2.axhline(700, c='red')
ax2.axhline(150, c='red')
ax2.axvline(600, c='red')
ax2.axvline(850, c='red')
ax2.set_xlabel('X [m]')
ax2.set_ylabel('Y [m]')

The index can also be reset but the old `index` column can be kept if needed for further processing.

The index can also be kept when setting `reset_index=False`. 

<a id='ex2'></a>
# Example 2 - LineString Data

## Load Data

As a second example, we are loading a shape file containing line information. Additional columns are `id`, which was not recorded when digitizing the data and `formation` indicating the base of the formation that was encountered at a particular point. This will become important in future tutorials. The term `Ton` is German for clay and indicates in this example the boundary between a sandstone layer and a claystone layer. In this example, the previously mentioned `Sand1` is also present. The same attributes can be inspected again as before. 

In [ ]:
gdf = gpd.read_file('../data/tutorial01/interfaces1_lines.shp')
gdf.head()

In [ ]:
gdf.length

In [ ]:
gdf.crs

In [ ]:
len(gdf)

In [ ]:
gdf.geometry[:5]

In [ ]:
gdf.geom_type

In [ ]:
gdf.bounds.head()

In [ ]:
gdf.total_bounds

In [ ]:
gdf.is_valid

As mentioned above, every entry of the `GeoDataFrame` is basically a shapely object, in this case a LineString:

In [ ]:
gdf.loc[0].geometry

In [ ]:
type(gdf.loc[0].geometry)

## Plotting Data

The data can be plotted using the built-in `GeoPandas` plotting functions and by adding additional `Matplotlib` commands like grids or axes labels.

In [ ]:
gdf.plot(aspect='equal')
plt.grid()
plt.xlabel('X [m]')
plt.ylabel('Y [m]')

## Extracting XY Coordinates

For the geomodeling with `GemPy` or other packages, the X and Y coordinates of the `shapely` LineStrings need to be extracted. The extraction of the X and Y values for LineStrings is done using list comprehension and the built-in `Pandas` `explode()` method. The functionality of extracting the X and Y coordinates is also combined in the function `extract_xy_linestrings`. 

The signature of the function is shown below:

```Signature:
gg.vector.extract_xy_linestrings(
    gdf: geopandas.geodataframe.GeoDataFrame,
    reset_index: bool = True,
    drop_id: bool = True,
    drop_index: bool = True,
    drop_points: bool = True,
    overwrite_xy: bool = False,
    target_crs: str = None,
    bbox: Union[Sequence[float], NoneType] = None,
) -> geopandas.geodataframe.GeoDataFrame
Docstring:
Extracting x,y coordinates from a GeoDataFrame (LineStrings) and returning a GeoDataFrame with x,y
coordinates as additional columns
Args:
    gdf (gpd.geodataframe.GeoDataFrame): GeoDataFrame created from vector data containing elements of geom_type
    LineString
    reset_index (bool): Variable to reset the index of the resulting GeoDataFrame, default True
    drop_id (bool): Variable to drop the id column, default True
    drop_index (bool): Variable to drop the index column, default True
    drop_points (bool): Variable to drop the points column, default True
    overwrite_xy (bool): Variable to overwrite existing X and Y values, default False
    target_crs (str, pyproj.crs.crs.CRS): Name of the CRS provided to reproject coordinates of the GeoDataFrame
    bbox (list): Values (minx, maxx, miny, maxy) to limit the extent of the data
Return:
    gdf (gpd.geodataframe.GeoDataFrame): GeoDataFrame with appended x,y columns and optional columns```

First, we are using list comprehension to create a `Pandas` `series` containing list of tuples containing the X and Y position of each vertex in each line. 

In [ ]:
gdf['points'] = [list(i.coords) for i in gdf.geometry]
print(type(gdf['points']))
gdf['points']

In [ ]:
gdf['points'][0][:5]

The lists within each row of `gdf['points']` now need to be exploded into single rows. In the `geometry` column, each row is now represented by a LineString which will be changed later to a Point object. The exploded point tuples are now present as additional rows.

In [ ]:
df = pd.DataFrame(gdf).explode('points')
df.head(10)

The `points` column is now converted into `X` and `Y` columns. 

In [ ]:
df[['X', 'Y']] = pd.DataFrame(df['points'].tolist(), index=df.index)
df.head(10)

Finally, the Pandas `DataFrame` is converted to a GeoPandas `GeoDataFrame`. The geometry column now consists of points instead of lines and is matching the `X` and `Y` columns. `id` and `index` columns as well as the `points` column are also present.

In [ ]:
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.X, df.Y), crs=gdf.crs)
gdf.head()

In [ ]:
gdf.reset_index()

## Extracting X and Y coordinates using GemGIS
All the steps above including the dropping of the additional columns are summarized in the function `extract_xy_linestrings`. The index is reset by default. This needs to be set to `False` for some applications introduced later. The `id`, `index` and `points` columns are dropped by default but arguments can also be set to false. The coordinates can also be reprojected as before by passing a `target_crs`. The extracted points can also be clipped by providing a `bbox`. 

In [ ]:
gdf = gpd.read_file('../data/tutorial01/interfaces1_lines.shp')
gdf.head()

In [ ]:
gdf_coordinates = gg.vector.extract_xy_linestrings(gdf)
gdf_coordinates.head()

<a id='ex3'></a>
# Example 3 - Polygon Data -> MultiLineStrings -> LineStrings

## Load Data

As a third and last example, we are loading a shape file containing polygon information. Additional columns are `id`, which was not recorded when digitizing the data and `formation` indicating the base of the formation that was encountered at a particular point. This will become important in future tutorials. The term `Ton` is German for clay and indicates in this example the boundary between a sandstone layer and a claystone layer. In this example, the previously mentioned `Sand1` is also present as well as `Sand2`. The same attributes can be inspected again as before. 

In [ ]:
gdf = gpd.read_file('../data/tutorial01/geolmap1.shp')
gdf.head()

In [ ]:
gdf.area

In [ ]:
gdf.crs

In [ ]:
len(gdf)

In [ ]:
gdf.geometry[:5]

In [ ]:
gdf.geom_type

In [ ]:
gdf.bounds.head()

In [ ]:
gdf.total_bounds

In [ ]:
gdf.is_valid

As mentioned above, every entry of the `GeoDataFrame` is basically a shapely object, in this case a Polygon:

In [ ]:
gdf.loc[0].geometry

In [ ]:
type(gdf.loc[0].geometry)

## Plotting Data

The data can be plotted using the built-in `GeoPandas` plotting functions and by adding additional `Matplotlib` commands like grids or axes labels. By passing the argument `column='formation'`, we can color the different Polygons according to their associated formation.

In [ ]:
gdf.plot(aspect='equal', column='formation')
plt.grid()
plt.xlabel('X [m]')
plt.ylabel('Y [m]')

## Extracting XY Coordinates

For the geomodeling with `GemPy` or other packages, the X and Y coordinates of the `shapely` polygons need to be extracted. The extraction of the X and Y values for polygons consists of several steps:

- Exploding Polygons to MultiLineStrings and LineStrings
- Exploding MultiLineStrings to LineStrings
- Extracting Coordinates from LineStrings

Each step will now be introduced.

## Exploding Polygons to MultiLineStrings and LineStrings

The first step is to explode the Polygons to MultiLineStrings and LineStrings. Since the original data does not contain any MultiLineStrings, a MultiLineString is created manually from the two LineStrings belonging to the formation `Sand2`.

In [ ]:
gdf_linestrings = gpd.GeoDataFrame(gdf.drop('geometry', axis=1), geometry=gdf.boundary, crs=gdf.crs)
gdf_linestrings

A new GeoDataFrame with a MultiLineString is created.

In [ ]:
from shapely.geometry import MultiLineString
gdf2 = gpd.GeoSeries(data = MultiLineString([gdf_linestrings.loc[2].geometry,gdf_linestrings.loc[3].geometry])).to_frame()
gdf2['formation']='Sand2'
gdf2['id'] = None
gdf2.columns = ['geometry','formation', 'id' ]
gdf2

The original and the new GeoDataFrame are concated.

In [ ]:
gdf_linestring = pd.concat([gdf_linestrings, gdf2]).reset_index()
gdf_linestring

The two duplicated rows for the formation `Sand2` are dropped. 

In [ ]:
gdf_linestring = gdf_linestring.drop(gdf_linestring.index[[2,3]]).reset_index()
gdf_linestring = gdf_linestring[['geometry','formation', 'id']]
gdf_linestring 

This step can also be done with the function `explode_polygons`, which will return in our case only LineStrings as indicated before.

In [ ]:
gdf_exploded = gg.vector.explode_polygons(gdf)
gdf_exploded

## Exploding MultiLineStrings to LineStrings

The second step is to explode MultiLineStrings to LineStrings. Therefore, we will use the manually created GeoDataFrame containing one MultiLineString to demonstrate the functionality. This can easily be done using the built-in `explode()` method of Pandas. However, when doing that, we will get two index columns.

In [ ]:
gdf_linestrings_exploded = gdf_linestring.explode()
gdf_linestrings_exploded

Using `explode_multilinestrings` will automatically explode all LineStrings and reset the index by default. 

In [ ]:
gdf_linestrings_exploded = gg.vector.explode_multilinestrings(gdf_linestring)
gdf_linestrings_exploded

Setting `reset_index=False` will create the table as shown above.

In [ ]:
gdf_linestrings_exploded = gg.vector.explode_multilinestrings(gdf_linestring, reset_index=False)
gdf_linestrings_exploded

By default, the additionally created columns `level0` and `level1` when resetting the index are dropped. However, they can also be kept.

In [ ]:
gdf_linestrings_exploded = gg.vector.explode_multilinestrings(gdf_linestring, reset_index=True, drop_level0=False, drop_level1=False)
gdf_linestrings_exploded

Now that we only have LineStrings in the GeoDataFrame, we can use the function `extract_xy_linestrings` to extract the X and Y coordinates. By default, the `level0` and `level1` columns are dropped but can be kept setting the dropping arguments to `False`. 

In [ ]:
gdf_xy = gg.vector.extract_xy_linestrings(gdf_linestrings_exploded)
gdf_xy

<a id='ex4'></a>
# Extract XY from any type of Geometry

**All introduced functions are combined to the function `extract_xy(...)` taking the loaded vector data as GeoDataFrame as input and the introduced arguments to drop columns, reproject the coordinates or clip the coordinates with a bounding box.**

<a id='summary'></a>
# Summary

In this tutorial it was shown how to 
- Extract XY coordinates from points using `extract_xy_points()`
- Extract XY coordinates from linestrings using `extract_xy_linestrings()`
- Explode Polygons using `explode_polygons()`
- Explode MultiLineStrings using `explode_multilinestrings()`
- Explode all types of Geometry using `extract_xy()`

In the next tutorial, the functionality to extract height information from a raster will be introduced.


In [ ]:
gdf.loc[0].geometry.bounds